# 02 · LLM Answer Generation
**Grounded Conversational Chart Accessibility** — Section 2 of 6

**What this notebook does, in plain language**

1. Loads the 200 questions saved by notebook 01.
2. Sends each question, with its chart data, to **two LLMs**:
   - **Llama 3.2 3B**: open-source, running free on your laptop through Ollama,
   - **Gemini Flash-Lite**: commercial, through Google's free tier.
3. Asks each question in **two styles**: a *short* answer ("62") and a *full sentence* ("Canada has the highest value, at 62").
4. Saves every answer to `outputs/llm_answers.csv` for Section 3 to verify.

Nothing is judged here. This notebook only *collects* answers. Checking them is Section 3's job.

**Every response is cached** in `cache/llm_responses.jsonl`. If the run stops halfway (laptop sleeps, free-tier limit hit), just run it again: finished answers load instantly and only the missing ones are requested.

## Before you start (one-time setup)

**Ollama:** make sure the Ollama app is running (its icon is in the Windows system tray). You already have `llama3.2:3b`.

**Gemini key:**
1. Get a free key at **aistudio.google.com** → *Get API key*.
2. Save it as an environment variable. In a terminal, run (with your real key):
   ```
   setx GEMINI_API_KEY "paste-your-key-here"
   ```
3. **Close and reopen VS Code completely**, so it picks up the new variable.

The key never appears in the code or the notebook, so it's safe to share the project.

## 2.1 Setup

In [1]:
import sys, logging
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src import config
from src.llm import (build_prompt, ask, ResponseCache, QuotaExhausted,
                     ollama_ready, gemini_ready)

config.make_dirs()
config.set_seeds()
logging.basicConfig(filename=config.LOG_DIR / "02_llm.log", filemode="a",
                    level=logging.INFO, format="%(levelname)s | %(message)s", force=True)

sample = pd.read_csv(config.OUTPUT_DIR / "eval_sample.csv")
cache = ResponseCache()
print(f"{len(sample)} questions loaded | {len(cache.data)} responses already cached")

200 questions loaded | 0 responses already cached


## 2.2 What the LLM actually sees

The prompt has four parts: the **chart type**, the **raw CSV data** (exactly as in ChartQA, messy symbols included), the **question**, and a **style rule** (short or sentence). A fixed *system prompt* also tells the model it is answering for blind and low-vision users, using only the given data.

`temperature = 0` makes answers as repeatable as possible.

In [2]:
ex = sample.iloc[0]
print(build_prompt(ex["query"], ex["table_file"], ex["chart_type"], "short"))

Chart type: line chart
Chart data (CSV):
Characteristic,Total consumption in million liters
2018,13779.1
2017,13260.9
2016,13294.5
2015,13131.8
2014,13149.3
2013,13213.1

Question: How many liters of non-alcoholic beverages were consumed in the UK in 2018?

Reply with ONLY the answer: a single number, name, or Yes/No. No units, no explanation.


## 2.3 Check which models are ready

Any model that isn't ready is skipped with a message telling you why, so you can run Llama alone first and add Gemini later.

In [10]:
active = []
for m in config.MODELS:
    ok = ollama_ready(m["model"]) if m["provider"] == "ollama" else gemini_ready()
    print(f"{m['name']:14s} {'READY' if ok else 'skipped'}")
    if ok:
        active.append(m)

llama3.2-3b    READY
gemini-flash   READY


## 2.4 The collection loop

For every question × model × style, `ask()` either returns the cached answer or calls the model. If the Gemini free tier says "no more today", the loop **stops cleanly**: everything so far is already saved, so just rerun tomorrow.

**Run a small test first:** leave `RUN_LIMIT = 5` and run this cell and the next one. Check the answers look sensible. Then set `RUN_LIMIT = None` and run again for all 200.

> **Timing:** Llama runs on your CPU, so expect a few seconds per answer. 400 Llama answers may take 20–40 minutes; let it run. Gemini pauses a few seconds between calls to respect the free tier.

In [7]:
RUN_LIMIT = None     # 5 for a quick test; None for all questions

def collect(questions, models, styles):
    rows = []
    total = len(questions) * len(models) * len(styles)
    done = 0
    for m in models:
        for style in styles:
            for q in questions.itertuples():
                prompt = build_prompt(q.query, q.table_file, q.chart_type, style)
                try:
                    rec = ask(m, prompt, cache)
                except QuotaExhausted:
                    print(f"\n{m['name']}: free-tier limit reached. Progress saved; rerun later to continue.")
                    break
                rows.append({"qid": q.qid, "model": m["name"], "style": style,
                             "llm_answer": rec["answer"], "error": rec["error"],
                             "latency_s": rec["latency_s"], "cached": rec["cached"]})
                done += 1
                if done % 20 == 0 or done == total:
                    print(f"  {done}/{total} done")
    return pd.DataFrame(rows)

subset = sample if RUN_LIMIT is None else sample.head(RUN_LIMIT)
answers = collect(subset, active, config.ANSWER_STYLES)
print(f"\nCollected {len(answers)} answers | errors: {answers['error'].notna().sum()}")

  20/800 done
  40/800 done
  60/800 done
  80/800 done
  100/800 done
  120/800 done
  140/800 done
  160/800 done
  180/800 done
  200/800 done
  220/800 done
  240/800 done
  260/800 done
  280/800 done
  300/800 done
  320/800 done
  340/800 done
  360/800 done
  380/800 done
  400/800 done
  420/800 done
  440/800 done
  460/800 done
  480/800 done
  500/800 done
  520/800 done
  540/800 done
  560/800 done
  580/800 done
  600/800 done
  620/800 done
  640/800 done
  660/800 done
  680/800 done
  700/800 done
  720/800 done
  740/800 done
  760/800 done
  780/800 done
  800/800 done

Collected 800 answers | errors: 0


## 2.5 Save and take a first look

We join the answers back to the question details and save them. The table below shows answers next to the gold label, just to eyeball them. Real verification happens in Section 3.

In [8]:
llm_answers = sample.merge(answers, on="qid")
llm_answers.to_csv(config.OUTPUT_DIR / "llm_answers.csv", index=False)
print(f"Saved {len(llm_answers)} rows -> outputs/llm_answers.csv")

llm_answers[["model", "style", "qtype", "query", "label", "llm_answer"]].head(20)

Saved 800 rows -> outputs/llm_answers.csv


,model,style,qtype,query,label,llm_answer
0,llama3.2-3b,short,lookup,How many liters of non-alcoholic beverages wer...,13779.1,"13,791.1"
1,llama3.2-3b,sentence,lookup,How many liters of non-alcoholic beverages wer...,13779.1,The total consumption of non-alcoholic beverag...
2,gemini-flash,short,lookup,How many liters of non-alcoholic beverages wer...,13779.1,13779.1
3,gemini-flash,sentence,lookup,How many liters of non-alcoholic beverages wer...,13779.1,"In 2018, the total consumption of non-alcoholi..."
4,llama3.2-3b,short,compare,What is the ratio of commercial to unknown?,2.625,1.6
5,llama3.2-3b,sentence,compare,What is the ratio of commercial to unknown?,2.625,The ratio of commercial to unknown seizures is...
6,gemini-flash,short,compare,What is the ratio of commercial to unknown?,2.625,2.625
7,gemini-flash,sentence,compare,What is the ratio of commercial to unknown?,2.625,The ratio of commercial to unknown counterfeit...
8,llama3.2-3b,short,diff,What is the difference between the highest and...,0.2,2.07
9,llama3.2-3b,sentence,diff,What is the difference between the highest and...,0.2,The difference between the highest and lowest ...


In [9]:
# How many answers per model and style, and how fast (only freshly-called answers have real timings)
summary = (llm_answers.groupby(["model", "style"])
           .agg(answers=("llm_answer", lambda s: s.notna().sum()),
                errors=("error", lambda s: s.notna().sum()),
                avg_seconds=("latency_s", "mean")).round(2))
summary

answers  errors  avg_seconds
model        style                                 
gemini-flash sentence      200       0         1.73
             short         200       0         1.59
llama3.2-3b  sentence      200       0         4.72
             short         200       0         4.22

## ✅ Checkpoint before Section 3

1. With `RUN_LIMIT = None`, the summary shows **200 answers** per model and style (400 per model), with few or no errors.
2. Read ~10 answers in each style. Short answers should be just a value; sentence answers should be one clear sentence.
3. `outputs/llm_answers.csv` exists. Check `outputs/logs/02_llm.log` if there were errors.